# gempipe documentation

gempipe is a tool for the reconstruction of genome-scale metabolic models (GSMMs or GEMs).
It generates models: 

* strain-specific and species-specific.
* with or without a reference model.
* compliant with the BiGG nomenclature.
* with annotated metabolites and reactions.

gempipe is divided in 3 parts: 

* **Part 1.** Creation of the presence/absence matrix (PAM) and the draft pan-model, starting from the genomes (or proteomes).
* **Part 2.** The draft pan-model is curated by the user with some heleper functions. 
* **Part 3.** Derivation of strain-specific and species-specific models, starting from the PAM and pan-model. 

Unlike other tools like [CarveMe](https://carveme.readthedocs.io/en/latest/index.html), we force the user to perform a manual curation. This will limit the number of modeled false-positive reactions. 

[Installation](page_1.ipynb)
[Part 1: gempipe recon](page_2.ipynb)
[Part 2: manual gapfilling](page_3.ipynb)
[Part 3: gempipe derive](page_4.ipynb)
[How to cite](page_5.ipynb)

[API documentation](autoapi/index)

In [552]:
import uuid


from IPython.display import display, HTML


class Pipeflow:
    def __init__(self, diagram):
        self.diagram = self.process_diagram(diagram)
        self.uid = uuid.uuid4()

    def process_diagram(self, diagram):
        diagram = diagram.replace("\n", "\\n")
        diagram = diagram.lstrip("\\n")
        diagram = diagram.replace("'", '"')
        return diagram

    def render(self, height=400, panzoom='true'):        
        html = f"""
        <style> #outcellbox {{display: flex; justify-content: center; width: 100%; height: {height}px; background-color: #ffffff;}} </style>
        <style> #outcellbox svg {{width: 100%; height: 100%;}} </style>
        <div class="mermaid-{self.uid}" id="outcellbox"></div>
        <script scr="https://github.com/bumbu/svg-pan-zoom/raw/3.6.1/src/svg-pan-zoom.js"></script>
        <script type="module">
            import mermaid from 'https://cdn.jsdelivr.net/npm/mermaid@10.6.1/+esm';
            const graphDefinition = \'___diagram___\';
            const element = document.querySelector('.mermaid-{self.uid}');
            const {{ svg }} = await mermaid.render('graphDiv-{self.uid}', graphDefinition);
            element.innerHTML = svg;
            var panZoomEnabler = svgPanZoom('#graphDiv-{self.uid}', 
                {{controlIconsEnabled: {panzoom}, panEnabled: {panzoom}, zoomEnabled: {panzoom}, dblClickZoomEnabled: false}});
        </script>
        """
        html = html.replace("___diagram___", self.diagram)
        return display(HTML(html))



#from gempipe import Pipeflow


gempipe_flow = Pipeflow("""
flowchart LR;


i1{{Input_proteomes}} --> Proteomes --> BRH
i2{{Input_genomes}} --> initial_Genomes
i3{{Input_taxids}} --> NCBI_download --> initial_Genomes{{initial_Genomes}}
i4{{Ref_proteome}} --> BRH
i5{{Ref_model}} --> BRH --> transl_Ref_model{{transl_Ref_model}}
i6{{Exp_setting}} --> Ref_expansion

initial_Genomes --> CDS_prediction --> initial_Proteomes{{initial_Proteomes}} --> Bio_metrics --> Busco(Busco)
initial_Genomes --> Tech_metrics --> N50(N50) & n_contigs(n_contigs)
N50 & n_contigs & Busco & initial_Genomes & initial_Proteomes --> Filtering 
Filtering --> Genomes{{Genomes}} & Proteomes{{Proteomes}}

Genomes & Proteomes --> Masking --> Masked_genomes{{Masked_genomes}}
Proteomes --> Clustering --> initial_PAM{{initial_PAM}} & initial_Ref_seqs{{initial_Ref_seqs}}
initial_PAM & initial_Ref_seqs & Proteomes & Genomes & Masked_genomes --> Gene_recovery

subgraph Gene_recovery
r1[Module_1] --> r2[Module_2] --> r3[Module_3]
end

Gene_recovery --> PAM{{PAM}} & Ref_seqs{{Ref_seqs}}
Ref_seqs --> Func_annot --> Gene_functions{{Gene_functions}} --> PruneU
BiGG_genes{{BiGG_genes}} & Ref_seqs & Universe{{Universe}} --> PruneU --> RF_draft_panmodel{{RF_draft_panmodel}}

RF_draft_panmodel & transl_Ref_model --> Ref_expansion --> draft_panmodel{{draft_panmodel}}


draft_panmodel --> Manual_curation((Manual_curation)) --> panmodel{{panmodel}}
panmodel & PAM --> strain_models{{strain_models}} --> gapfilling --> gf_strain_models{{gf_strain_model}} --> species_models{{species_models}}
    
style i1 fill:lightgreen
style i2 fill:lightgreen
style i3 fill:lightgreen
style i4 fill:lightgreen
style i5 fill:lightgreen
style i6 fill:lightgreen

style initial_Genomes fill:lightblue
style Masked_genomes fill:lightblue   
style Busco fill:lightsalmon
style N50 fill:lightsalmon
style n_contigs fill:lightsalmon
style Genomes fill:lightblue
style initial_Proteomes fill:lightblue
style Proteomes fill:lightblue
style initial_PAM fill:lightblue
style initial_Ref_seqs fill:lightblue
style PAM fill:lightblue
style Ref_seqs fill:lightblue
style BiGG_genes fill:violet
style Universe fill:violet
style RF_draft_panmodel fill:lightblue
style transl_Ref_model fill:lightblue
style draft_panmodel fill:lightblue
style panmodel fill:gold
style strain_models fill:lightblue
style gf_strain_models fill:gold
style species_models fill:gold
style Gene_functions fill:violet


style NCBI_download fill:whitesmoke
style CDS_prediction fill:whitesmoke
style Bio_metrics fill:whitesmoke
style Tech_metrics fill:whitesmoke
style Clustering fill:whitesmoke
style Masking fill:whitesmoke
style r1 fill:whitesmoke
style r2 fill:whitesmoke
style r3 fill:whitesmoke
style BRH fill:whitesmoke
style PruneU fill:whitesmoke
style Ref_expansion fill:whitesmoke
style Manual_curation fill:whitesmoke
style gapfilling fill:whitesmoke
style Func_annot fill:whitesmoke

""")

Below we report the outline of gempipe. 
* green nodes: files provided by the user (note that gempipe can start from either genomes, proteomes, or taxids).
* blue nodes: intermediated files generated.
* gold nodes: main output files.
* pink nodes: internal databases, automatically handled. 

In [553]:
# Work in progress!
gempipe_flow.render()